In [ ]:
import matplotlib as mpl

TEXTWIDTH_PT = 418.25368
TEXTWIDTH_IN = TEXTWIDTH_PT / 72.27

def setup_pub_style(fontsize=9):
    mpl.rcParams.update({
        "font.size": fontsize,
        "axes.titlesize": fontsize,
        "axes.labelsize": fontsize,
        "xtick.labelsize": fontsize - 1,
        "ytick.labelsize": fontsize - 1,
        "legend.fontsize": fontsize - 1,
        "figure.dpi": 300,
        "savefig.dpi": 300,
    })

def fig_textwidth(height_ratio=0.62):
    import matplotlib.pyplot as plt
    return plt.subplots(figsize=(TEXTWIDTH_IN, TEXTWIDTH_IN * height_ratio))

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# ---- load image (replace with your file) ----
img = cv2.imread("polar_bear2.png")
img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# ---- 1. Harris corners ----
harris = cv2.cornerHarris(np.float32(img_gray), blockSize=2, ksize=3, k=0.04)
harris = cv2.dilate(harris, None)
harris_img = img_rgb.copy()
harris_img[harris > 0.01 * harris.max()] = [255, 0, 0]  # red

# ---- 2. Shi–Tomasi (Good Features to Track) ----
shi = cv2.goodFeaturesToTrack(
    img_gray,
    maxCorners=300,
    qualityLevel=0.01,
    minDistance=10
)
shi_img = img_rgb.copy()
if shi is not None:
    for p in shi:
        x, y = p.ravel().astype(int)
        cv2.circle(shi_img, (x, y), 3, (0, 255, 0), -1)  # green

# ---- 3. SIFT keypoints ----
sift = cv2.SIFT_create()
kp = sift.detect(img_gray, None)
sift_img = cv2.drawKeypoints(
    img_rgb, kp, None,
    flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
)

# ---- plot results ----
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(harris_img)
axes[0].set_title("Harris corners")
axes[1].imshow(shi_img)
axes[1].set_title("Shi-Tomasi features")
axes[2].imshow(sift_img)
axes[2].set_title("SIFT keypoints")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

IMG_PATH = "polar_bear2.png"

PALETTE = {
    "blue":     "#0072B2",
    "orange":   "#E69F00",
    "skyblue":  "#56B4E9",
    "green":    "#009E73",
    "yellow":   "#F0E442",
    "vermilion":"#D55E00",
    "purple":   "#CC79A7",
    "black":    "#000000",
}

def hex_to_rgb(h):
    h = h.lstrip("#")
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))

HARRIS_COLOR = hex_to_rgb(PALETTE["vermilion"])
EDGE_COLOR   = hex_to_rgb(PALETTE["yellow"])
SIFT_COLOR   = hex_to_rgb(PALETTE["purple"])

img_bgr = cv2.imread(IMG_PATH)
if img_bgr is None:
    raise FileNotFoundError(f"Could not read image: {IMG_PATH}")

img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# Harris
harris = cv2.cornerHarris(np.float32(img_gray), 2, 3, 0.04)
harris = cv2.dilate(harris, None, iterations=2)

harris_img = img_rgb.copy()
thr = 0.02 * harris.max()
ys, xs = np.where(harris > thr)

# limit density for readability
max_pts = 600
if len(xs) > max_pts:
    idx = np.random.choice(len(xs), max_pts, replace=False)
    xs, ys = xs[idx], ys[idx]

for x, y in zip(xs, ys):
    cv2.circle(harris_img, (int(x), int(y)), 3, HARRIS_COLOR, -1)

# Canny 
edges = cv2.Canny(img_gray, threshold1=100, threshold2=200)

edges_img = img_rgb.copy()
edge_mask = edges > 0

kernel = np.ones((4, 4), np.uint8)  
edges_thick = cv2.dilate(edges, kernel, iterations=1)
edges_img[edges_thick > 0] = EDGE_COLOR

# SIFT 
sift = cv2.SIFT_create()
kp = sift.detect(img_gray, None)

kp = sorted(kp, key=lambda k: -k.response)[:300]

# sift_img = cv2.drawKeypoints(
#     img_rgb,
#     kp,
#     None,
#     color=SIFT_COLOR,
#     flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
# )
sift_img = img_rgb.copy()

for k in kp:
    x, y = map(int, k.pt)
    r = max(int(k.size / 2), 4)
    angle = np.deg2rad(k.angle)

    # circle (scale)
    cv2.circle(sift_img, (x, y), r, SIFT_COLOR, 3)

    # orientation line
    x2 = int(x + r * np.cos(angle))
    y2 = int(y + r * np.sin(angle))
    cv2.line(sift_img, (x, y), (x2, y2), SIFT_COLOR, 3)


plt.rcParams["figure.dpi"] = 150

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(harris_img)
axes[0].set_title("Harris corners")
axes[1].imshow(edges_img)
axes[1].set_title("Canny edges")
axes[2].imshow(sift_img)
axes[2].set_title("SIFT keypoints (scale + orientation)")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

# fig.savefig("feature_detectors_corner_edge_blob.png", dpi=300, bbox_inches="tight")

In [ ]:
import cv2
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

IMG_PATH = "polar_bear2.png"

PALETTE = {
    "blue":     "#0072B2",
    "orange":   "#E69F00",
    "skyblue":  "#56B4E9",
    "green":    "#009E73",
    "yellow":   "#F0E442",
    "vermilion":"#D55E00",
    "purple":   "#CC79A7",
    "black":    "#000000",
}

def hex_to_rgb(h):
    h = h.lstrip("#")
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))

HARRIS_COLOR = hex_to_rgb(PALETTE["vermilion"])
EDGE_COLOR   = hex_to_rgb(PALETTE["yellow"])
SIFT_COLOR   = hex_to_rgb(PALETTE["purple"])

# --- publication-style formatting (your template) ---
TEXTWIDTH_PT = 418.25368
TEXTWIDTH_IN = TEXTWIDTH_PT / 72.27

def setup_pub_style(fontsize=9):
    mpl.rcParams.update({
        "font.size": fontsize,
        "axes.titlesize": fontsize,
        "axes.labelsize": fontsize,
        "xtick.labelsize": fontsize - 1,
        "ytick.labelsize": fontsize - 1,
        "legend.fontsize": fontsize - 1,
        "figure.dpi": 300,
        "savefig.dpi": 300,
    })

def fig_textwidth(height_ratio=0.62):
    return plt.subplots(figsize=(TEXTWIDTH_IN, TEXTWIDTH_IN * height_ratio))

setup_pub_style(fontsize=9)
# ----------------------------------------------------

img_bgr = cv2.imread(IMG_PATH)
if img_bgr is None:
    raise FileNotFoundError(f"Could not read image: {IMG_PATH}")

img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# Harris
harris = cv2.cornerHarris(np.float32(img_gray), 2, 3, 0.04)
harris = cv2.dilate(harris, None, iterations=2)

harris_img = img_rgb.copy()
thr = 0.02 * harris.max()
ys, xs = np.where(harris > thr)

# limit density for readability
max_pts = 600
if len(xs) > max_pts:
    rng = np.random.default_rng(0)  # reproducible
    idx = rng.choice(len(xs), max_pts, replace=False)
    xs, ys = xs[idx], ys[idx]

for x, y in zip(xs, ys):
    cv2.circle(harris_img, (int(x), int(y)), 3, HARRIS_COLOR, -1)

# Canny
edges = cv2.Canny(img_gray, threshold1=100, threshold2=200)

edges_img = img_rgb.copy()
kernel = np.ones((4, 4), np.uint8)
edges_thick = cv2.dilate(edges, kernel, iterations=1)
edges_img[edges_thick > 0] = EDGE_COLOR

# SIFT
sift = cv2.SIFT_create()
kp = sift.detect(img_gray, None)
kp = sorted(kp, key=lambda k: -k.response)[:300]

sift_img = img_rgb.copy()
for k in kp:
    x, y = map(int, k.pt)
    r = max(int(k.size / 2), 4)
    angle = np.deg2rad(k.angle)

    cv2.circle(sift_img, (x, y), r, SIFT_COLOR, 3)
    x2 = int(x + r * np.cos(angle))
    y2 = int(y + r * np.sin(angle))
    cv2.line(sift_img, (x, y), (x2, y2), SIFT_COLOR, 3)

# --- plotting using textwidth-based figure sizing ---
# 3 panels side-by-side -> choose a wide ratio
fig, axes = fig_textwidth(height_ratio=0.35)
# override to fit 3 columns while still using textwidth as the base
fig.set_size_inches(TEXTWIDTH_IN, TEXTWIDTH_IN * 0.35)

axes = np.atleast_1d(axes)
# If fig_textwidth returns a single Axes, replace with 1x3 layout at textwidth
plt.close(fig)
fig, axes = plt.subplots(1, 3, figsize=(TEXTWIDTH_IN, TEXTWIDTH_IN * 0.35))

axes[0].imshow(harris_img)
axes[0].set_title("Harris corners")
axes[1].imshow(edges_img)
axes[1].set_title("Canny edges")
axes[2].imshow(sift_img)
axes[2].set_title("SIFT blobs")

for ax in axes:
    ax.axis("off")

plt.tight_layout(pad=0.3, w_pad=0.4)
plt.show()

fig.savefig("polarbear_features.pdf", bbox_inches="tight")

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

IMG_PATH = "polar_bear2.png"

# ---- colorblind-safe palette (Okabe–Ito / seaborn-style) ----
def hex_to_rgb(h):
    h = h.lstrip("#")
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))
HARRIS_COLOR = hex_to_rgb(PALETTE["vermilion"])
EDGE_COLOR   = hex_to_rgb(PALETTE["yellow"])
SIFT_COLOR   = hex_to_rgb(PALETTE["purple"])

# ---- knobs ----
PATCH_RADIUS = 35   # patch will be (2R+1) x (2R+1)
DOT_RADIUS = 4
THICKNESS = 2

# ---- load ----
img_bgr = cv2.imread(IMG_PATH)
if img_bgr is None:
    raise FileNotFoundError(f"Could not read image: {IMG_PATH}")
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
H, W = img_gray.shape

def crop_patch(img, x, y, r=PATCH_RADIUS):
    x1, x2 = max(0, x-r), min(W, x+r+1)
    y1, y2 = max(0, y-r), min(H, y+r+1)
    return img[y1:y2, x1:x2], (x1, y1)

# ----------------------------
# 1) Harris corners -> pick strongest corner away from borders
# ----------------------------
harris = cv2.cornerHarris(np.float32(img_gray), 2, 3, 0.04)
harris = cv2.dilate(harris, None, iterations=2)

margin = PATCH_RADIUS + 2
harris_valid = harris.copy()
harris_valid[:margin, :] = -np.inf
harris_valid[-margin:, :] = -np.inf
harris_valid[:, :margin] = -np.inf
harris_valid[:, -margin:] = -np.inf

y_c, x_c = np.unravel_index(np.argmax(harris_valid), harris_valid.shape)

# ----------------------------
# 2) Canny edges -> pick an edge pixel away from borders
# ----------------------------
edges = cv2.Canny(img_gray, 100, 200)
edge_ys, edge_xs = np.where(edges > 0)

# filter by margin to allow patch crop
ok = (edge_xs > margin) & (edge_xs < W-margin) & (edge_ys > margin) & (edge_ys < H-margin)
edge_xs, edge_ys = edge_xs[ok], edge_ys[ok]
if len(edge_xs) == 0:
    raise RuntimeError("No edge pixels found (try lower Canny thresholds).")

# choose a representative edge pixel (here: random from valid)
idx = np.random.randint(len(edge_xs))
x_e, y_e = int(edge_xs[idx]), int(edge_ys[idx])

# ----------------------------
# 3) SIFT blobs -> pick strongest keypoint away from borders
# ----------------------------
sift = cv2.SIFT_create()
kp = sift.detect(img_gray, None)
kp = [k for k in kp if margin < k.pt[0] < W-margin and margin < k.pt[1] < H-margin]
kp = sorted(kp, key=lambda k: -k.response)
if len(kp) == 0:
    raise RuntimeError("No SIFT keypoints found.")
k_best = kp[0]
x_b, y_b = map(int, k_best.pt)
r_b = max(int(k_best.size / 3), 6)
angle = np.deg2rad(k_best.angle)

# ----------------------------
# Make overview image with markers
# ----------------------------
overview = img_rgb.copy()

# corner marker
cv2.circle(overview, (x_c, y_c), DOT_RADIUS+1, HARRIS_COLOR, THICKNESS)

# edge marker
cv2.circle(overview, (x_e, y_e), DOT_RADIUS+1, EDGE_COLOR, THICKNESS)

# blob marker (circle + orientation line)
cv2.circle(overview, (x_b, y_b), r_b, SIFT_COLOR, THICKNESS)
x2 = int(x_b + r_b * np.cos(angle))
y2 = int(y_b + r_b * np.sin(angle))
cv2.line(overview, (x_b, y_b), (x2, y2), SIFT_COLOR, THICKNESS)

# ----------------------------
# Crop patches and overlay local markers
# ----------------------------
corner_patch, (cx1, cy1) = crop_patch(img_rgb, x_c, y_c)
edge_patch,   (ex1, ey1) = crop_patch(img_rgb, x_e, y_e)
blob_patch,   (bx1, by1) = crop_patch(img_rgb, x_b, y_b)

corner_patch_viz = corner_patch.copy()
cv2.circle(corner_patch_viz, (x_c - cx1, y_c - cy1), DOT_RADIUS, HARRIS_COLOR, THICKNESS)

edge_patch_viz = edge_patch.copy()
# show local edges in patch too
edges_patch, _ = crop_patch(edges, x_e, y_e)
edge_mask = edges_patch > 0
edge_patch_viz[edge_mask] = EDGE_COLOR
cv2.circle(edge_patch_viz, (x_e - ex1, y_e - ey1), DOT_RADIUS, EDGE_COLOR, THICKNESS)

blob_patch_viz = blob_patch.copy()
cv2.circle(blob_patch_viz, (x_b - bx1, y_b - by1), r_b, SIFT_COLOR, THICKNESS)
x2p = int((x_b - bx1) + r_b * np.cos(angle))
y2p = int((y_b - by1) + r_b * np.sin(angle))
cv2.line(blob_patch_viz, (x_b - bx1, y_b - by1), (x2p, y2p), SIFT_COLOR, THICKNESS)

# ----------------------------
# Plot: overview + 3 close-ups
# ----------------------------
plt.rcParams["figure.dpi"] = 150
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].imshow(overview)
axes[0, 0].set_title("Overview (selected corner, edge, blob)")
axes[0, 1].imshow(corner_patch_viz)
axes[0, 1].set_title("Close-up: corner (Harris)")

axes[1, 0].imshow(edge_patch_viz)
axes[1, 0].set_title("Close-up: edge (Canny)")

axes[1, 1].imshow(blob_patch_viz)
axes[1, 1].set_title("Close-up: blob (SIFT)")

for ax in axes.ravel():
    ax.axis("off")

plt.tight_layout()
plt.show()